# Kaggle Remote Training Notebook

Run this notebook with a Kaggle Jupyter Server kernel from VS Code/Colab.
It prepares SH17 + Pictor data and launches training with minimal friction.

In [11]:
from pathlib import Path
import os
import subprocess
import sys

print('Python:', sys.version)
print('CWD:', Path.cwd())
print('Kaggle input exists:', Path('/kaggle/input').exists())
print('Kaggle working exists:', Path('/kaggle/working').exists())

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
CWD: /kaggle/working
Kaggle input exists: True
Kaggle working exists: True


## Configure dataset paths

Set these to your mounted Kaggle dataset folders under `/kaggle/input`.

In [12]:
# Update these names to match your Kaggle dataset mount points.
# Based on your /kaggle/input listing:
#  - /kaggle/input/datasets/mugheesahmad/sh17-dataset-for-ppe-detection
#  - /kaggle/input/datasets/zyanahmed/pictor-ppe
SH17_DIR = Path('/kaggle/input/datasets/mugheesahmad/sh17-dataset-for-ppe-detection')
PICTOR_DIR = Path('/kaggle/input/datasets/zyanahmed/pictor-ppe')


print('SH17_DIR:', SH17_DIR, SH17_DIR.exists())
print('PICTOR_DIR:', PICTOR_DIR, PICTOR_DIR.exists())


SH17_DIR: /kaggle/input/datasets/mugheesahmad/sh17-dataset-for-ppe-detection True
PICTOR_DIR: /kaggle/input/datasets/zyanahmed/pictor-ppe True


In [14]:
# Dependencies (Internet ON)
cmd = [
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    'ultralytics',
    'kagglehub',
    'pyyaml',
    'pillow',
    'tqdm',
]

print('Running:', ' '.join(cmd))
subprocess.check_call(cmd)

Running: /usr/bin/python3 -m pip install -q ultralytics kagglehub pyyaml pillow tqdm
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.2 MB/s eta 0:00:00


0

In [15]:
# Ensure the repo code exists on the Kaggle VM (under /kaggle/working).
# If you attached this repo as a Kaggle Dataset, this will copy it from /kaggle/input.
# Otherwise (with Internet ON) it will git clone.

REPO_DST = Path('/kaggle/working/cross-factory-tta')


def has_repo(p: Path) -> bool:
    return (p / 'src' / 'train.py').exists()


def find_repo_in_kaggle_input() -> Path | None:
    base = Path('/kaggle/input')
    if not base.exists():
        return None

    # Search a couple of levels deep for something that looks like this repo.
    # This avoids walking huge dataset trees.
    candidates = []
    for p in base.glob('*'):
        candidates.append(p)
        candidates.extend(list(p.glob('*')))
        candidates.extend(list(p.glob('*/*')))

    for p in candidates:
        if p.is_dir() and has_repo(p):
            return p
    return None


if has_repo(Path.cwd()):
    REPO_ROOT = Path.cwd()
    print('Repo already in CWD:', REPO_ROOT)
else:
    src_repo = find_repo_in_kaggle_input()
    if src_repo is not None:
        print('Found repo in /kaggle/input:', src_repo)
        REPO_DST.mkdir(parents=True, exist_ok=True)
        # Copy code into writable /kaggle/working
        subprocess.check_call(['bash', '-lc', f"cp -r '{src_repo}/'* '{REPO_DST}/'"])
    else:
        # Fallback: clone from GitHub (requires Internet ON)
        if not REPO_DST.exists() or not any(REPO_DST.iterdir()):
            print('Repo not found in /kaggle/input; cloning into /kaggle/working...')
            subprocess.check_call([
                'git', 'clone',
                '--depth', '1',
                'https://github.com/rainsfal1/cross-factory-tta.git',
                str(REPO_DST),
            ])

    REPO_ROOT = REPO_DST

os.chdir(REPO_ROOT)
print('Repo root:', Path.cwd())
print('Has src/train.py:', (Path.cwd() / 'src' / 'train.py').exists())

AssertionError: Open this notebook from repo root or cd into repo first.

In [ ]:
# Prepare data into repo-local data/ folder.
prep_cmd = [
    sys.executable,
    '-u',
    'src/prepare_data.py',
    '--sh17-dir',
    str(SH17_DIR),
    '--pictor-dir',
    str(PICTOR_DIR),
]
print('Running:', ' '.join(prep_cmd))
subprocess.check_call(prep_cmd)

## Smoke run (fast sanity check)

In [ ]:
smoke_cmd = [
    sys.executable,
    '-u',
    'src/train.py',
    '--data',
    'data/sh17/sh17.yaml',
    '--model',
    'yolov8m.pt',
    '--epochs',
    '1',
    '--imgsz',
    '320',
    '--batch',
    '4',
    '--device',
    '0',
    '--name',
    'sh17_smoke_remote',
    '--project',
    '/kaggle/working/runs/train',
]
print('Running:', ' '.join(smoke_cmd))
subprocess.check_call(smoke_cmd)

## Full training run

In [ ]:
train_cmd = [
    sys.executable,
    '-u',
    'src/train.py',
    '--data',
    'data/sh17/sh17.yaml',
    '--model',
    'yolov8m.pt',
    '--epochs',
    '50',
    '--imgsz',
    '640',
    '--batch',
    '16',
    '--device',
    '0',
    '--name',
    'sh17_yolov8m',
    '--project',
    '/kaggle/working/runs/train',
]
print('Running:', ' '.join(train_cmd))
subprocess.check_call(train_cmd)